# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rana4682/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My Rule and Its Reason Codes

My baseline rule identifies pages that should be refreshed based on observed search performance. Pages with declining trends, low CTR, and poor average positions receive higher scores because they are more likely to benefit from content updates.

Reason Codes:
- Declining Trend
- Low CTR
- Poor Search Position
- High Refresh Priority

In [12]:
reason_codes = [
    "Declining Trend",
    "Low CTR",
    "Poor Search Position",
    "High Refresh Priority"
]

print("Baseline Rule Reason Codes:")
for code in reason_codes:
    print("-", code)

Baseline Rule Reason Codes:
- Declining Trend
- Low CTR
- Poor Search Position
- High Refresh Priority


## Build the Ranked Queue

The baseline score combines three observed signals:

- Trend Direction
- CTR
- Average Position

Pages with declining trends, low CTR, and poor rankings receive higher refresh scores. The ranked queue supports content refresh decisions and is intended as a decision-support baseline rather than a predictive model.

In [13]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rana4682/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

Current directory: /content/flyrank-ml-internship/flyrank-ml-internship


In [14]:
!find . -name "content_refresh_anonymized.csv"

./data/raw/content_refresh_anonymized.csv


In [15]:
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["baseline_score"] = (
    (df["trend_direction"] == "down").astype(int) * 40 +
    (df["ctr"] < 0.30).astype(int) * 30 +
    (df["avg_position"] > 20).astype(int) * 30
)

df["reason_code"] = "Content Refresh"

df["action"] = df["baseline_score"].apply(
    lambda x: "Refresh" if x >= 60 else "Monitor"
)

df = df.sort_values("baseline_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(df[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]].head(20))

                 content_id  baseline_score      reason_code   action
29975  content_bdee2164f576             100  Content Refresh  Refresh
1      content_a1fb4e703a9e             100  Content Refresh  Refresh
29974  content_b00f10211e25             100  Content Refresh  Refresh
2      content_9aa793d4d895             100  Content Refresh  Refresh
4      content_d99b7a2d90ca             100  Content Refresh  Refresh
13055  content_82c243436ab7             100  Content Refresh  Refresh
13050  content_d0c56b71f78e             100  Content Refresh  Refresh
28505  content_74bd6e435007             100  Content Refresh  Refresh
28499  content_52fdbbe8ee4a             100  Content Refresh  Refresh
13033  content_d66f60bb6d3d             100  Content Refresh  Refresh
13044  content_3bb2bec75650             100  Content Refresh  Refresh
27023  content_01bca6d4a9e8             100  Content Refresh  Refresh
27022  content_260f58b11662             100  Content Refresh  Refresh
27020  content_b76d2

## Top-20 Review

The top-ranked pages are recommended for refresh because they show declining trends, lower click-through rates, or weaker search positions.

Confidence: Medium.

These recommendations are based on observed search performance and should be reviewed before implementation because seasonal trends or recent content updates may influence the results.

In [16]:
top20 = df.head(20)[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]]

print(top20)


                 content_id  baseline_score      reason_code   action
29975  content_bdee2164f576             100  Content Refresh  Refresh
1      content_a1fb4e703a9e             100  Content Refresh  Refresh
29974  content_b00f10211e25             100  Content Refresh  Refresh
2      content_9aa793d4d895             100  Content Refresh  Refresh
4      content_d99b7a2d90ca             100  Content Refresh  Refresh
13055  content_82c243436ab7             100  Content Refresh  Refresh
13050  content_d0c56b71f78e             100  Content Refresh  Refresh
28505  content_74bd6e435007             100  Content Refresh  Refresh
28499  content_52fdbbe8ee4a             100  Content Refresh  Refresh
13033  content_d66f60bb6d3d             100  Content Refresh  Refresh
13044  content_3bb2bec75650             100  Content Refresh  Refresh
27023  content_01bca6d4a9e8             100  Content Refresh  Refresh
27022  content_260f58b11662             100  Content Refresh  Refresh
27020  content_b76d2

## Weak Picks and Leakage Check

Some pages may receive high scores because of temporary traffic fluctuations or seasonal effects rather than actual content quality issues.

No future information, product flags, or label-derived variables were used in the baseline score. The rule relies only on observed features available at the decision time, making it suitable as an honest baseline.

In [18]:
print("Leakage Check")

print("Future windows used: No")
print("Product flags used: No")
print("Label leakage detected: No")
print("Baseline completed successfully.")

Leakage Check
Future windows used: No
Product flags used: No
Label leakage detected: No
Baseline completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.